In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np
from IPython.display import display


In [ ]:
from data_processing.constants import DEFAULT_TICKERS
from data_analysis.loader import MarketDataLoader
from data_analysis.cleaner import MarketDataCleaner
from data_analysis.features import MarketFeatureEngineer

loader = MarketDataLoader()
raw_df = loader.load_analysis_data(DEFAULT_TICKERS)

print("Raw Shape:", raw_df.shape)
clean_df = MarketDataCleaner.clean(raw_df)
df = MarketFeatureEngineer.add_all_features(clean_df)


In [ ]:
from data_analysis.analysis import(
    correlation_matrix,
    asset_class_correlation,
    annual_performance,
    monthly_performance
)

## Instrument correlaiton matrix

In [ ]:
instrument_corr = correlation_matrix(df)

plt.figure(figsize=(12, 10))
sns.heatmap(instrument_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Instrument Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Instrument Correlation Matrix:")
display(instrument_corr.round(2))

## Asset class correlation

In [ ]:
asset_corr = asset_class_correlation(df)

plt.figure(figsize=(10, 8))
sns.heatmap(asset_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Asset Class Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Asset Class Correlation Matrix:")
display(asset_corr.round(2))

##  Annual performance

In [ ]:
annual = annual_performance(df)

plt.figure(figsize=(14, 6))
annual.plot(kind='bar', ax=plt.gca(), width=0.8)
plt.title('Annual Performance by Instrument', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Return (%)')
plt.legend(loc='best', bbox_to_anchor=(1.05, 1), borderaxespad=0)
plt.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Annual Performance:")
display(annual.round(2))

## Monthly performance

In [ ]:
monthly = monthly_performance(df)

plt.figure(figsize=(14, 8))
monthly.plot(kind='line', marker='o', ax=plt.gca(), alpha=0.7)
plt.title('Monthly Performance by Instrument', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Return (%)')
plt.legend(loc='best', bbox_to_anchor=(1.05, 1), borderaxespad=0)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Monthly Performance Summary:")
display(monthly.describe().round(2))

## Rolling volatility over time

In [ ]:
rolling_window = 30
volatility = df.pct_change().rolling(window=rolling_window).std() * np.sqrt(252)

plt.figure(figsize=(14, 6))
for col in volatility.columns:
    plt.plot(volatility.index, volatility[col], label=col, alpha=0.7)
plt.title('Rolling 30-Day Volatility (Annualized)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Rolling Volatility Statistics:")
display(volatility.describe().round(4))